# Grand Mobile · классификатор категорий (этап D handoff)

Замороженный multilingual-E5 + логистическая регрессия против baseline (регулярки репо) и char TF-IDF.
Опционально — дообучение E5 на двух T4 (DataParallel). Выбор модели — только по **dev**; **test** (разметка людьми)
считается один раз в конце. Данные приватные: Kaggle dataset держать **Private**, вывод не публиковать.

**Запуск:** Add Data → свой приватный dataset с `gm_nlp_data.jsonl`; Settings → Accelerator **GPU T4 x2**, **Internet On**; Run All.

In [1]:
import os, json, glob, random, hashlib, time
import numpy as np, pandas as pd, torch
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModel

SMOKE = os.environ.get("GM_SMOKE") == "1"          # local CPU check only
SEED = 20261031
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_GPU = torch.cuda.device_count()
print("device:", DEVICE, "| GPUs:", N_GPU, [torch.cuda.get_device_name(i) for i in range(N_GPU)])

ENCODERS = (["intfloat/multilingual-e5-small", "intfloat/multilingual-e5-base", "intfloat/multilingual-e5-large"]
            if not SMOKE else [os.environ["GM_LOCAL_E5"]])
RUN_FINETUNE = not SMOKE                               # experiment: fine-tune e5-base on both GPUs

device: cuda | GPUs: 2 ['Tesla T4', 'Tesla T4']


In [2]:
path = os.environ.get("GM_DATA") or glob.glob("/kaggle/input/**/gm_nlp_data.jsonl", recursive=True)[0]
df = pd.read_json(path, lines=True)
DATA_SHA = hashlib.sha256(open(path, "rb").read()).hexdigest()[:16]
print(path, DATA_SHA); print(df.groupby(["split", "kind"]).size().unstack())

CATS = {"complaint": ["complaint.insult", "complaint.cheat", "complaint.dm", "complaint.scam", "complaint.rmt_ads",
                      "complaint.admin", "complaint.other", "complaint.invalid"],
        "report": ["report.question", "report.help", "report.complaint", "report.account", "report.payment", "report.offtopic"]}
LABELS = CATS["complaint"] + CATS["report"]
PRIOS = ["p0", "p1", "p2", "p3"]
# Default priority of each category (taxonomy criteria + team labelling rules).
DEFAULT_PRIO = {"complaint.insult": "p2", "complaint.cheat": "p1", "complaint.dm": "p2", "complaint.scam": "p1",
                "complaint.rmt_ads": "p1", "complaint.admin": "p2", "complaint.other": "p2", "complaint.invalid": "p3",
                "report.question": "p3", "report.help": "p2", "report.complaint": "p2", "report.account": "p2",
                "report.payment": "p0", "report.offtopic": "p3"}

def model_input(r):
    # Only what the service sees at inference: channel, the player's form field, player text.
    parts = [f"канал: {r.kind}"]
    if r.field: parts.append(f"пункт правил: {r.field}")
    parts.append(r.text[:2000])
    return " | ".join(parts)

df["input"] = [model_input(r) for r in df.itertuples()]
tr, dv, te = (df[df.split == s].reset_index(drop=True) for s in ("train", "dev", "test"))
print(len(tr), len(dv), len(te)); print(tr.category.value_counts())

/kaggle/input/datasets/naadgob/grand-mobile-train/gm_nlp_data.jsonl 237815f6567f3efd
kind   complaint  report
split                   
dev           82      18
test         161      37
train        376     192
568 100 198
category
complaint.insult     86
complaint.admin      84
complaint.dm         70
report.offtopic      45
report.help          41
complaint.invalid    39
report.complaint     30
complaint.scam       28
report.question      28
complaint.rmt_ads    25
report.payment       24
report.account       24
complaint.cheat      22
complaint.other      22
Name: count, dtype: int64


In [3]:
class Pooled(torch.nn.Module):
    """Mean-pooled, L2-normalised E5 sentence vector (the model card recipe)."""
    def __init__(self, name):
        super().__init__(); self.enc = AutoModel.from_pretrained(name)
    def forward(self, input_ids, attention_mask):
        h = self.enc(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        m = attention_mask.unsqueeze(-1).to(h.dtype)
        return torch.nn.functional.normalize((h * m).sum(1) / m.sum(1).clamp(min=1e-9), dim=-1)

@torch.inference_mode()
def embed(name, texts, batch=64):
    tok = AutoTokenizer.from_pretrained(name)
    net = Pooled(name).to(DEVICE).eval()
    if DEVICE == "cuda": net = net.half()
    if N_GPU > 1: net = torch.nn.DataParallel(net)            # batches split over both T4
    out = []
    for i in range(0, len(texts), batch):
        enc = tok(["query: " + t for t in texts[i:i + batch]], max_length=512, truncation=True,
                  padding=True, return_tensors="pt").to(DEVICE)
        out.append(net(enc["input_ids"], enc["attention_mask"]).float().cpu())
    del net; torch.cuda.empty_cache()
    return torch.cat(out).numpy()

EMB = {}
for name in ENCODERS:
    t0 = time.time()
    EMB[name] = {s: embed(name, d["input"].tolist()) for s, d in (("train", tr), ("dev", dv), ("test", te))}
    print(f"{name}: {EMB[name]['train'].shape} in {time.time() - t0:.0f}s")

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (29 outputs left out)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

intfloat/multilingual-e5-large: (568, 1024) in 21s


In [4]:
def masked_predict(proba, classes, kinds):
    """Arg-max restricted to the categories of the ticket's channel (never a cross-channel code)."""
    classes = list(classes); preds, conf = [], []
    for p, k in zip(proba, kinds):
        allowed = [i for i, c in enumerate(classes) if c.startswith(k + ".")]
        if not allowed:                                          # channel unseen in training
            preds.append(CATS[k][0]); conf.append(0.0); continue
        i = max(allowed, key=lambda j: p[j]); s = p[allowed].sum()
        preds.append(classes[i]); conf.append(float(p[i] / s) if s else 0.0)
    return np.array(preds), np.array(conf)

def scores(y, p, labels):
    return {"acc": round(accuracy_score(y, p), 3), "macro_f1": round(f1_score(y, p, labels=labels, average="macro", zero_division=0), 3)}

def fit_lr(Xtr, ytr, Xdv, ydv, kinds_dv, labels):
    best = None
    for C in (0.25, 1, 4, 16, 64):
        clf = LogisticRegression(C=C, max_iter=5000, class_weight="balanced").fit(Xtr, ytr)
        pred, _ = masked_predict(clf.predict_proba(Xdv), clf.classes_, kinds_dv) if labels is LABELS else (clf.predict(Xdv), None)
        s = scores(ydv, pred, sorted(set(ydv)))
        if best is None or s["macro_f1"] > best[1]["macro_f1"]: best = (clf, s, C)
    return best

results, heads = [], {}
present = sorted(set(tr.category) | set(dv.category))
# 1) baseline: the repository's regex heuristics (no training)
results.append({"model": "baseline_regex", **scores(dv.category, dv.baseline_category, present)})
# 2) char TF-IDF + LR
tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True, max_features=100_000)
Xtr_t, Xdv_t = tfidf.fit_transform(tr.input), tfidf.transform(dv.input)
clf, s, C = fit_lr(Xtr_t, tr.category, Xdv_t, dv.category, dv.kind, LABELS)
heads["tfidf"] = (clf, C); results.append({"model": f"tfidf_char_lr (C={C})", **s})
# 3) frozen E5 + LR, one per encoder
for name in ENCODERS:
    clf, s, C = fit_lr(EMB[name]["train"], tr.category, EMB[name]["dev"], dv.category, dv.kind, LABELS)
    heads[name] = (clf, C); results.append({"model": f"{name.split('/')[-1]}+lr (C={C})", **s})
pd.DataFrame(results).sort_values("macro_f1", ascending=False)

,model,acc,macro_f1
1,tfidf_char_lr (C=64),0.69,0.657
4,multilingual-e5-large+lr (C=16),0.62,0.630
3,multilingual-e5-base+lr (C=64),0.66,0.590
2,multilingual-e5-small+lr (C=64),0.60,0.575
0,baseline_regex,0.53,0.408


In [5]:
FT = None
if RUN_FINETUNE:
    # Experiment: full fine-tune of e5-base, DataParallel over both T4, fp16, early stop on dev macro-F1.
    name = "intfloat/multilingual-e5-base"
    tok = AutoTokenizer.from_pretrained(name)
    idx = {c: i for i, c in enumerate(LABELS)}
    class Clf(torch.nn.Module):
        def __init__(self):
            super().__init__(); self.pool = Pooled(name); self.head = torch.nn.Linear(self.pool.enc.config.hidden_size, len(LABELS))
        def forward(self, ids, mask): return self.head(self.pool(ids, mask))
    net = Clf().to(DEVICE); para = torch.nn.DataParallel(net) if N_GPU > 1 else net
    opt = torch.optim.AdamW(net.parameters(), lr=3e-5, weight_decay=0.01)
    scaler = torch.cuda.amp.GradScaler()
    counts = tr.category.value_counts(); w = torch.tensor([len(tr) / (len(counts) * counts.get(c, len(tr))) for c in LABELS], device=DEVICE)
    lossf = torch.nn.CrossEntropyLoss(weight=w.float())
    def batches(d, bs, shuffle):
        order = np.random.permutation(len(d)) if shuffle else np.arange(len(d))
        for i in range(0, len(d), bs):
            part = d.iloc[order[i:i + bs]]
            enc = tok(["query: " + t for t in part["input"]], max_length=256, truncation=True, padding=True, return_tensors="pt")
            yield enc["input_ids"].to(DEVICE), enc["attention_mask"].to(DEVICE), part
    @torch.inference_mode()
    def ft_predict(d):
        para.eval(); probs = []
        for ids, mask, _ in batches(d, 64, False):
            with torch.autocast("cuda", dtype=torch.float16): probs.append(torch.softmax(para(ids, mask).float(), -1).cpu())
        return masked_predict(torch.cat(probs).numpy(), LABELS, d.kind)
    best_f1, best_state, patience = -1, None, 0
    for epoch in range(8):
        para.train()
        for ids, mask, part in batches(tr, 32, True):
            y = torch.tensor([idx[c] for c in part.category], device=DEVICE)
            with torch.autocast("cuda", dtype=torch.float16): loss = lossf(para(ids, mask).float(), y)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        s = scores(dv.category, ft_predict(dv)[0], present); print("epoch", epoch, s)
        if s["macro_f1"] > best_f1:
            best_f1, patience = s["macro_f1"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
        else:
            patience += 1
            if patience >= 2: break
    net.load_state_dict(best_state); FT = ft_predict
    results.append({"model": "e5-base finetuned (2xT4)", **scores(dv.category, ft_predict(dv)[0], present)})
board = pd.DataFrame(results).sort_values("macro_f1", ascending=False).reset_index(drop=True); board

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_48/2047522233.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


epoch 0 {'acc': 0.14, 'macro_f1': 0.154}
epoch 1 {'acc': 0.32, 'macro_f1': 0.242}
epoch 2 {'acc': 0.39, 'macro_f1': 0.288}
epoch 3 {'acc': 0.44, 'macro_f1': 0.352}
epoch 4 {'acc': 0.48, 'macro_f1': 0.336}
epoch 5 {'acc': 0.48, 'macro_f1': 0.361}
epoch 6 {'acc': 0.58, 'macro_f1': 0.439}
epoch 7 {'acc': 0.52, 'macro_f1': 0.379}


,model,acc,macro_f1
0,tfidf_char_lr (C=64),0.69,0.657
1,multilingual-e5-large+lr (C=16),0.62,0.630
2,multilingual-e5-base+lr (C=64),0.66,0.590
3,multilingual-e5-small+lr (C=64),0.60,0.575
4,e5-base finetuned (2xT4),0.58,0.439
5,baseline_regex,0.53,0.408


In [6]:
# Choose on DEV only. Simpler model wins ties (fewer moving parts in production).
order = {"baseline_regex": 0, "tfidf": 1}
board["rank_key"] = [(-r.macro_f1, -r.acc, 0 if "small" in r.model else 1) for r in board.itertuples()]
winner = board.sort_values("rank_key").iloc[0].model
print("DEV winner:", winner)

def predict(model, d, split):
    if model == "baseline_regex": return d.baseline_category.values, np.ones(len(d))
    if model.startswith("tfidf"): clf = heads["tfidf"][0]; X = tfidf.transform(d.input)
    elif model.startswith("e5-base finetuned"): return FT(d)
    else:
        name = next(n for n in ENCODERS if model.startswith(n.split("/")[-1])); clf = heads[name][0]; X = EMB[name][split]
    return masked_predict(clf.predict_proba(X), clf.classes_, d.kind)

# Abstention threshold on DEV: lowest confidence that keeps >= 90% accuracy on the kept tickets.
p_dv, c_dv = predict(winner, dv, "dev"); THRESH = 0.0
for t in np.round(np.arange(0.0, 0.95, 0.05), 2):
    keep = c_dv >= t
    if keep.sum() >= 10 and accuracy_score(dv.category[keep], p_dv[keep]) >= 0.9: THRESH = float(t); break
print("abstention threshold (dev, not calibrated probability):", THRESH, "| dev coverage:", round((c_dv >= THRESH).mean(), 3))

DEV winner: tfidf_char_lr (C=64)
abstention threshold (dev, not calibrated probability): 0.5 | dev coverage: 0.62


In [7]:
# Priority: default-by-category vs a learned head, chosen on DEV.
def prio_from_cat(cats): return np.array([DEFAULT_PRIO[c] for c in cats])
p_dv_cat, _ = predict(winner, dv, "dev")
prio_res = {"default_by_category": scores(dv.priority, prio_from_cat(p_dv_cat), PRIOS),
            "baseline_regex": scores(dv.priority, dv.baseline_priority, PRIOS)}
PRIO_HEAD = None
if winner.startswith(tuple(n.split("/")[-1] for n in ENCODERS)) and not winner.startswith("e5-base finetuned"):
    name = next(n for n in ENCODERS if winner.startswith(n.split("/")[-1]))
    PRIO_HEAD = LogisticRegression(C=heads[name][1], max_iter=5000, class_weight="balanced").fit(EMB[name]["train"], tr.priority)
    prio_res["learned_head"] = scores(dv.priority, PRIO_HEAD.predict(EMB[name]["dev"]), PRIOS)
prio_winner = max(prio_res, key=lambda k: prio_res[k]["macro_f1"])
print(pd.DataFrame(prio_res).T); print("priority DEV winner:", prio_winner)

                      acc  macro_f1
default_by_category  0.84     0.562
baseline_regex       0.72     0.469
priority DEV winner: default_by_category


In [8]:
# ===== FINAL: test is used exactly once, after every choice above was made on dev =====
p_te, c_te = predict(winner, te, "test")
if prio_winner == "default_by_category": pr_te = prio_from_cat(p_te)
elif prio_winner == "learned_head": pr_te = PRIO_HEAD.predict(EMB[name]["test"])
else: pr_te = te.baseline_priority.values
labels_te = sorted(set(te.category))
final = pd.DataFrame([
    {"model": "baseline_regex", **scores(te.category, te.baseline_category, labels_te),
     "priority_macro_f1": scores(te.priority, te.baseline_priority, PRIOS)["macro_f1"]},
    {"model": winner, **scores(te.category, p_te, labels_te), "priority_macro_f1": scores(te.priority, pr_te, PRIOS)["macro_f1"]}])
print(f"TEST n={len(te)} (human labels)"); display(final)
print(classification_report(te.category, p_te, labels=labels_te, zero_division=0))
te["pred"] = p_te; te["lang_bucket"] = te.lang_hint.where(te.lang_hint.isin(["ru", "uz", "kk"]), "other")
display(te.groupby("lang_bucket").apply(lambda g: pd.Series({"n": len(g), "acc": round((g.pred == g.category).mean(), 3)})))
keep = c_te >= THRESH
print(f"with abstention t={THRESH}: coverage {keep.mean():.2f}, accuracy on kept {accuracy_score(te.category[keep], p_te[keep]):.3f}")
cm = pd.DataFrame(confusion_matrix(te.category, p_te, labels=labels_te), index=labels_te, columns=labels_te); cm

TEST n=198 (human labels)


,model,acc,macro_f1,priority_macro_f1
0,baseline_regex,0.515,0.441,0.450
1,tfidf_char_lr (C=64),0.687,0.563,0.531


                   precision    recall  f1-score   support

  complaint.admin       0.82      0.69      0.75        59
  complaint.cheat       0.60      0.75      0.67         4
     complaint.dm       0.72      0.79      0.75        42
 complaint.insult       0.79      0.81      0.80        37
complaint.invalid       0.17      0.33      0.22         6
  complaint.other       0.00      0.00      0.00         4
   complaint.scam       0.88      0.78      0.82         9
 report.complaint       0.17      0.17      0.17         6
      report.help       0.91      0.62      0.74        16
  report.offtopic       0.39      0.58      0.47        12
  report.question       1.00      0.67      0.80         3

         accuracy                           0.69       198
        macro avg       0.58      0.56      0.56       198
     weighted avg       0.72      0.69      0.70       198



/tmp/ipykernel_48/3385620525.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  display(te.groupby("lang_bucket").apply(lambda g: pd.Series({"n": len(g), "acc": round((g.pred == g.category).mean(), 3)})))


,n,acc
lang_bucket,,
kk,6.0,0.333
other,23.0,0.609
ru,145.0,0.752
uz,24.0,0.458


with abstention t=0.5: coverage 0.70, accuracy on kept 0.804


,complaint.admin,complaint.cheat,complaint.dm,complaint.insult,complaint.invalid,complaint.other,complaint.scam,report.complaint,report.help,report.offtopic,report.question
complaint.admin,41,0,8,7,3,0,0,0,0,0,0
complaint.cheat,0,3,0,0,1,0,0,0,0,0,0
complaint.dm,4,1,33,1,1,1,1,0,0,0,0
complaint.insult,0,0,2,30,4,1,0,0,0,0,0
complaint.invalid,2,1,1,0,2,0,0,0,0,0,0
complaint.other,2,0,1,0,1,0,0,0,0,0,0
complaint.scam,1,0,1,0,0,0,7,0,0,0,0
report.complaint,0,0,0,0,0,0,0,1,1,4,0
report.help,0,0,0,0,0,0,0,0,10,6,0
report.offtopic,0,0,0,0,0,0,0,5,0,7,0


In [9]:
# Artifact for backend/app/classifier.py (local, versioned; never commit to Git).
import joblib
art = {"version": f"gm-cat-{DATA_SHA}", "winner": winner, "labels": LABELS, "channel_labels": CATS,
       "default_priority": DEFAULT_PRIO, "priority_method": prio_winner, "abstain_below": THRESH,
       "input_format": "query: канал: <kind> | пункт правил: <field> | <text[:2000]>", "data_sha256_16": DATA_SHA,
       "metrics_test": final.to_dict("records"), "confidence_kind": "dev_thresholded_uncalibrated"}
if winner.startswith("tfidf"): art |= {"kind": "tfidf_lr", "vectorizer": tfidf, "head": heads["tfidf"][0]}
elif winner.startswith("e5-base finetuned"): art |= {"kind": "finetuned", "encoder": "intfloat/multilingual-e5-base"}
elif winner != "baseline_regex":
    art |= {"kind": "frozen_e5_lr", "encoder": name, "head": heads[name][0], "priority_head": PRIO_HEAD}
os.makedirs("/kaggle/working" if not SMOKE else "smoke_out", exist_ok=True)
out_dir = "/kaggle/working" if not SMOKE else "smoke_out"
joblib.dump(art, f"{out_dir}/gm_category_model.joblib")
if winner.startswith("e5-base finetuned"): torch.save(net.state_dict(), f"{out_dir}/gm_e5base_finetuned.pt")
json.dump({k: v for k, v in art.items() if isinstance(v, (str, int, float, list, dict))}, open(f"{out_dir}/gm_model_card.json", "w"), ensure_ascii=False, indent=1, default=str)
print("saved to", out_dir, "| winner:", winner)

saved to /kaggle/working | winner: tfidf_char_lr (C=64)
